In [2]:
import os
import re
from functools import total_ordering
from pathlib import Path
from typing import Self
from functools import total_ordering

from attr import dataclass, astuple, asdict
from github.Requester import Requester
from github import GithubIntegration

from dulwich.repo import Repo
from dulwich import porcelain

from version_flow import git
from version_flow.clairity_repo import ClairityRepo
from version_flow.project_config import ProjectConfig
# from version_flow.project_version import ProjectVersionFda as Version
from version_flow.project_version import ProjectVersion
from version_flow.types import BumpPriority


In [3]:
p = Path("~/projects/clarity/terraform-aws-clinical-data-pipeline").expanduser()
repo = Repo(p.as_posix())

In [4]:
rc = repo.get_config()

for key in rc.keys():
    print(f"{key} => {rc[key]}")

(b'core',) => <dulwich.config.CaseInsensitiveOrderedMultiDict object at 0x1106040e0>
(b'remote', b'origin') => <dulwich.config.CaseInsensitiveOrderedMultiDict object at 0x110605a60>
(b'branch', b'main') => <dulwich.config.CaseInsensitiveOrderedMultiDict object at 0x110605c40>
(b'user',) => <dulwich.config.CaseInsensitiveOrderedMultiDict object at 0x110605eb0>


In [5]:
rc = repo.get_config()

for key, val in rc.items((b'remote', b'origin')):
    print(f"{key} => {val}")

b'url' => b'git@github.com:clairity-inc/terraform-aws-clinical-data-pipeline.git'
b'fetch' => b'+refs/heads/*:refs/remotes/origin/*'


In [6]:
for key, val in rc.items((b'user',)):
    print(f"{key} => {val}")

b'name' => b'Christian Bonnell'


In [7]:
rc.get((b'remote', b'origin'), b'url')

b'git@github.com:clairity-inc/terraform-aws-clinical-data-pipeline.git'

In [8]:
repo.get_config_stack().get((b'remote', b'origin'), b'url')

b'git@github.com:clairity-inc/terraform-aws-clinical-data-pipeline.git'

In [9]:
for key in repo.get_config_stack().sections():
    print(key)

(b'core',)
(b'remote', b'origin')
(b'branch', b'main')
(b'user',)


In [20]:
p = Path(".").expanduser()
p

PosixPath('.')

In [23]:
list(p.glob("*"))

[PosixPath('project_version.py,cover'),
 PosixPath('message_parsing.py'),
 PosixPath('fda_flows.py,cover'),
 PosixPath('project_version.py'),
 PosixPath('git.py'),
 PosixPath('message_parsing.py,cover'),
 PosixPath('types.py,cover'),
 PosixPath('trunk_flow.py,cover'),
 PosixPath('project_config.py,cover'),
 PosixPath('workbench.ipynb'),
 PosixPath('version.py'),
 PosixPath('fda_flows.py'),
 PosixPath('__init__.py'),
 PosixPath('message.py,cover'),
 PosixPath('version.py,cover'),
 PosixPath('project_config.py'),
 PosixPath('types.py'),
 PosixPath('__pycache__'),
 PosixPath('cli.py,cover'),
 PosixPath('trunk_flow.py'),
 PosixPath('alt_version_workbench.ipynb'),
 PosixPath('git.py,cover'),
 PosixPath('clairity_repo.py'),
 PosixPath('cli.py'),
 PosixPath('errors.py,cover'),
 PosixPath('auc_report_generator_debug.ipynb'),
 PosixPath('repo_config_workbench.ipynb'),
 PosixPath('.ipynb_checkpoints'),
 PosixPath('errors.py'),
 PosixPath('clairity_repo.py,cover'),
 PosixPath('workbench.py,cover'

In [11]:
refs_heads = repo.refs.as_dict(b"refs/remotes/origin")
refs_heads

{b'dependabot/pip/python/pytest-8.3.5': b'acaa8c6d1b1c2b201bf80f1e7178a41a4c39e2f0',
 b'main': b'28433c5c9288a4a98bb02799942687dbc61d7af8',
 b'HEAD': b'28433c5c9288a4a98bb02799942687dbc61d7af8',
 b'dependabot/pip/python/aenum-3.1.16': b'e6cc8746b2e59fba0dc8805b7fa6853f15f9b991',
 b'dependabot/pip/python/pip-25.1': b'd006f290c105db21f419a23a4a4243bf1c421dd7',
 b'PS-3258_fix_filter_negative_exams': b'09f4c40dbd73da97ef1eee17bf1fdd3c43add6bd',
 b'dependabot/pip/python/pytest-cov-6.1.1': b'af605fec7b33a2abc5e2c0edb2942d018e970d47',
 b'dependabot/pip/python/boto3-stubs-lite-1.38.17': b'4d0596c884f1d695ea2ba5b72567b92fa3bc53aa',
 b'HOTFIX-deploy-fersion-flow-hotfix': b'80b439ea9c51aeed1a9eeecddb1c2fa8556b94ec',
 b'PS-3258_filter_negative_exams': b'1deec10ceeca9d6bf90d9db93afd38e924a7aa19',
 b'dependabot/pip/python/pydantic-1.10.22': b'7e320ca45d9422204f5e0f365a02e8129e8d2837',
 b'dependabot/pip/python/pip-25.1.1': b'e6e0868e117df5540dc3af1b3901c884e47772b2',
 b'dependabot/pip/python/boto3-st

In [12]:
len(set(refs_heads.keys())), len(set(refs_heads.values()))

(21, 20)

In [13]:
bnames = git.commit_id_to_branch_name_map(repo)

AttributeError: module 'version_flow.git' has no attribute 'commit_id_to_branch_name_map'

In [32]:
{k: v for k,v in bnames.items() if len(v) > 1}

{b'3a89c3e8ec51637bee73cb3cf1d3b667d1bcf01c': [b'HEAD', b'main']}

In [33]:
porcelain.active_branch(repo)

b'main'

In [ ]:
porcelain.checkout_branch()

In [34]:
head_commit = repo[repo.head()]
head_commit.message

b'chore(ci): updating to version v1.11.5-rc.0 [skip ci]'

In [35]:
merge_commit = repo[head_commit.parents[0]]
print(merge_commit.message.decode())

Merge pull request #217 from clairity-inc/dependabot/pip/python/pytest-cov-6.1.1

build(deps-dev): bump pytest-cov from 6.0.0 to 6.1.1 in /python


In [66]:
re.match(r"Merge pull request #(?P<pr_number>\d+) ", merge_commit.message.decode()).groupdict()

{'pr_number': '217'}

In [36]:
merge_commit.parents

[b'61941d4eb41dc7759b9d098559cecb0b0322c5a0',
 b'31559a298220cab923969c979b9c1a3ffb99db43']

In [37]:
bnames[b'61941d4eb41dc7759b9d098559cecb0b0322c5a0'], bnames[b'31559a298220cab923969c979b9c1a3ffb99db43']

([], [])

In [38]:
bnames[repo.head()]

[b'HEAD', b'main']

In [39]:
# Problem:  It is very common for branches to be deleted before version-flow even finished running 
# in CircleCI. This means that we can't rely on the branch ref (b"refs/heads/branch-name") to 
# determine what branch was merged. This is unfortunate, because the only other way to get the two
# branch names is to text mine the commit message of the merge commit, as generated by GitHub. This
# is at least semi-structured, but relying on that structure to remain unchanging is not the best
# design option.

In [67]:
from github import Github, Auth
from github.Branch import Branch
from github.PullRequest import PullRequest
from github.Repository import Repository as GHRepository

In [41]:
token = os.environ["GH_TOKEN"]
auth = Auth.Token(token)
gh = Github(auth=auth)
gh

In [68]:
gh_repo = gh.get_repo("clairity-inc/terraform-aws-clinical-data-pipeline")
gh_repo

Repository(full_name="clairity-inc/terraform-aws-clinical-data-pipeline")

In [90]:
gh_repo.owner

NamedUser(login="clairity-inc")

In [93]:
user = gh.get_user("clairity-inc")
user

NamedUser(login="clairity-inc")

In [94]:
user.id

85695557

In [74]:
# pulls = gh_repo.get_pulls(state="closed", sort="updated", direction="desc")
pulls = gh_repo.get_pulls(state="closed")

In [75]:
n_printed = 0

for pull in pulls:
    print(f"{pull.number} ==> {pull.title} || {pull.merged_at}")
    n_printed += 1
    
    if n_printed > 20:
            break

220 ==> build(deps): bump pip from 25.0.1 to 25.1 in /python || None
219 ==> feat: PS-3258 Implement filter_negative_exams || 2025-05-01 15:02:31+00:00
218 ==> build(deps): bump aenum from 3.1.15 to 3.1.16 in /python || 2025-05-01 15:07:31+00:00
217 ==> build(deps-dev): bump pytest-cov from 6.0.0 to 6.1.1 in /python || 2025-05-01 15:45:33+00:00
216 ==> build(deps): bump pydantic from 1.10.21 to 1.10.22 in /python || 2025-05-01 15:12:11+00:00
215 ==> build(deps): bump s3pathlib from 2.3.1 to 2.3.2 in /python || 2025-05-01 15:35:22+00:00
214 ==> build(deps-dev): bump moto from 5.0.28 to 5.1.4 in /python || 2025-05-01 15:27:34+00:00
213 ==> HOTFIX deploy version-flow hotfix || 2025-04-25 15:19:30+00:00
212 ==> ci: update version-flow to v0.11.1 || 2025-04-25 13:15:22+00:00
211 ==> build(deps): bump boto3-stubs-lite from 1.36.21 to 1.38.2 in /python || 2025-04-25 03:24:59+00:00
210 ==> build(deps): bump boto3-stubs-lite from 1.36.21 to 1.38.1 in /python || None
209 ==> feat: PS-3258 Add ge

In [71]:
n_printed = 0

for pull in pulls:
    print(f"{pull.number} ==> {pull.base}")
    n_printed += 1
    
    if n_printed > 30 or pull.number == 199:
            break

220 ==> PullRequestPart(sha="3a89c3e8ec51637bee73cb3cf1d3b667d1bcf01c")
217 ==> PullRequestPart(sha="52faf2a990f9a32e3f88e38265e76504058fcd60")
215 ==> PullRequestPart(sha="d34b13c9c1b7042489f5cd5148a61799fedc67fc")
214 ==> PullRequestPart(sha="3e6d4ee083acbafc620900df7a9cd4e1e54936c9")
216 ==> PullRequestPart(sha="1dab504a470b8876ad6712e482fc1e7c6bbb5426")
218 ==> PullRequestPart(sha="1dab504a470b8876ad6712e482fc1e7c6bbb5426")
219 ==> PullRequestPart(sha="1712f1a32d58380bd6c068392de267a92883a3b9")
209 ==> PullRequestPart(sha="9dd24b712888c28a17e1b4a02e1f33d1f3071925")
213 ==> PullRequestPart(sha="6863e4f967aba7a6693678538060e2d00442f99b")
212 ==> PullRequestPart(sha="68b84f756fd9ff50c397f8b0fd046b4d733c06e3")
211 ==> PullRequestPart(sha="0e6103a0e89629174ea9e37bd434c2fa4e894448")
167 ==> PullRequestPart(sha="72c672ef3d55897f02a45cfeb4b6eb2d77dfce6a")
163 ==> PullRequestPart(sha="7589dfa2cffe05eb57fd4b5d853d396069a19112")
158 ==> PullRequestPart(sha="fabedd83f20a80235a6efcbc76c13425c34

In [76]:
pull

PullRequest(title="test: PS-3258 fixture to create DICOM metadata", number=200)

In [55]:
b = pull.base
b

PullRequestPart(sha="c037e5bcdf886d4d7fb84c86ec6b5623a6761ba5")

In [56]:
pull.base.ref

'main'

In [57]:
pull.head.ref

'PS-2903-fixes'

In [62]:
pull.user

NamedUser(login="shriramnarayanan")

In [50]:
# So it looks like our full approach is going to be:
# 1. Read the commit message from the last commit, and match "Merge pull request #nnn" to get the PR number
# 2. Scan the closed pull requests from github until we find the one with the correct PR number
# 3. Extract the branch names, knowing that we just merged pull.head.ref onto pull.base.ref

In [96]:
float("inf"), float("-inf")

(inf, -inf)

In [51]:
# Construction of a new version requires injecting information about the repository context
# into the bumping of a version, since we need to be able to (a) look up the name of the current
# branch (which requires access to the GitHub API), (b) figure out what label should be used for
# that branch name (which requires access to the ProjectConfig), and (c*) verify that there are no
# other branches which match the pattern that assigns that label (more info from dulwich.Repo).

# The design of this feature requires some thought about division of responsibility. Which classes
# should actually be responsible for what, and how do we ensure minimal coupling between them.

# I think the ProjectConfig and ClairityRepo class are already a bit big. And ClairityRepo is 
# probably bordering on having too many responsibilities. While ProjectSettings is becoming big, 
# it has at least stayed close to its original single responsibility of representing the settings
# of the pyproject.toml to whatever parts of the application that need them. (A good argument 
# could be made, though, that the "assign new version" should not be included in the class, and
# that the class should be made read-only.)

# One option is to put the version bumping logic in a VersionFactory class. In this case, the
# class that represents a version does not know how to bump itself, but only how to represent
# the data of a single version.

# Another option is to leave the bump logic in the version class, but to inject the label for
# the current branch into the bump function. But the current label is not needed for all types
# of bump, only for some. So then we have to either require the current label in all cases whether
# it is needed or not (not ideal), or only require it for the bump types that need it (which
# moves some of the bump logic outside the bump function and into the calling code, which is a
# violation of encapsulation).

# We have two different kinds of "release" branches... one that is a "named release" that should
# be treated as a release branch for the sake of deciding what types of merges are allowed, but
# has a normal release suffix with the version bump rules that apply. Then we also have the
# "production release" branch which has no suffix. It's version bump rules have to be slightly
# different than other versions.

# Note on c*: I'm not sure how easy it will be to enforce uniqueness of label because of branches 
# getting deleted. Hopefully, though, we are not actively assigning a version label to a branch 
# that has been deleted, so if we are assigning a label to an existent branch, we can at 
# least check all other branches to make sure there are no other branches using that label. 


# =======================================
# This requires a lot of dependency injection.
class VersionFactory:
    def __init__(self, config: ProjectConfig, repo: ClairityRepo):
        ...
    
    def from_string(self) -> ProjectVersion:
        ...
    
    def bump_version(self, current_version: ProjectVersion, bump_priority: BumpPriority) -> ProjectVersion:
        ...

conf = ProjectConfig("some/path/here")
repo = ClairityRepo(conf)
vfactory = VersionFactory(conf, repo)

current_version = vfactory.from_string(conf.version_string)

bump_priority = BumpPriority.patch
new_version = vfactory.bump_version(current_version, bump_priority)

conf.set_new_version(new_version)



ValueError: Cannot do anything with this config file: some/path/here

In [ ]:
# This requires a label to be looked up and injected in order to perform any version bump

class NewVersion:
    def __init__(self, major, minor, patch, label, revision):
        ...
    
    def bump(self, bump_priority: BumpPriority, current_branch_label: str) -> "NewVersion":
        ...

In [147]:
zde = ValueError("You divided something by zero!")

In [148]:
print(f"We got a: {zde}")

We got a: You divided something by zero!


In [149]:
print(f"We got a: {zde!r}")

We got a: ValueError('You divided something by zero!')


In [4]:
b"\xaa\x55".decode("latin-1")

'ªU'

In [6]:
b"\xDE\xAD\xBE".decode("latin-1")

'Þ\xad¾'

In [7]:
b"\xFE\xED\xFA".decode("latin-1")

'þíú'

In [10]:
b"\xF0\x0D".decode("unicode-escape"), b"\xF0\x0D".decode("latin-1")

('ð\r', 'ð\r')

In [12]:
import secrets
secrets.token_urlsafe(16)

'csVZFIHbgpZZRw9LZHVzZg'

In [ ]:
# old: ueb7wmk@twa4BZD_tnu
# new: csVZFIHbgpZZRw9LZHVzZg